# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedheshatthi104ff-hub/Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists("/content/Internship"):
    !git clone https://github.com/vedheshatthi104ff-hub/Internship.git /content/Internship
%cd /content/Internship
os.makedirs("work/outputs", exist_ok=True)

import pandas as pd, numpy as np, json
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")
df["declining"] = (df["trend_direction"] == "down").astype(int)
print(df.shape, "| overall decline rate:", round(df["declining"].mean(), 3))

Cloning into '/content/Internship'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 169 (delta 74), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 1.86 MiB | 14.02 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/Internship
(30000, 45) | overall decline rate: 0.542


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SIGNAL 1: staleness (behind FlyRank's refresh flags)
df["stale_bucket"] = pd.qcut(df["days_since_last_update"], 4, duplicates="drop")
t1 = (df.groupby("stale_bucket", observed=True)
        .agg(n=("declining", "size"), decline_rate=("declining", "mean")).round(3))
print("SIGNAL 1: staleness\n", t1, "\n")

# SIGNAL 2: CTR gap vs position tier (behind FlyRank's CTR-fix logic)
d2 = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 100)].copy()
d2["pos_tier"] = pd.cut(d2["avg_position"], [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
d2["expected_ctr"] = d2.groupby("pos_tier", observed=True)["ctr"].transform("median")
d2["ctr_gap"] = (d2["expected_ctr"] - d2["ctr"]) / d2["expected_ctr"]
d2["gap_bucket"] = pd.qcut(d2["ctr_gap"], 4, duplicates="drop")
t2 = (d2.groupby("gap_bucket", observed=True)
        .agg(n=("declining", "size"), decline_rate=("declining", "mean")).round(3))
print("SIGNAL 2: CTR gap vs position\n", t2)

SIGNAL 1: staleness
                     n  decline_rate
stale_bucket                       
(0.999, 20.0]   15866         0.539
(20.0, 104.0]   13816         0.546
(104.0, 373.0]    318         0.547 

SIGNAL 2: CTR gap vs position
                       n  decline_rate
gap_bucket                           
(-50.131, -1.087]  3801         0.526
(-1.087, 0.0]      3880         0.581
(0.0, 0.696]       3732         0.630
(0.696, 1.0]       3678         0.746


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
base = df[df["impressions_90d"] >= 100].copy()

base["demand"] = np.log1p(base["impressions_90d"]).rank(pct=True)
base["stale"] = base["days_since_last_update"].rank(pct=True)
base["pos_tier"] = pd.cut(base["avg_position"], [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
exp = base.groupby("pos_tier", observed=True)["ctr"].transform("median")
base["gap"] = ((exp - base["ctr"]) / exp).clip(0, 1).fillna(0)

W = {"demand": 0.40, "stale": 0.35, "gap": 0.25}     # change these if a verdict says so
base["score"] = (100 * (W["demand"]*base["demand"] + W["stale"]*base["stale"] + W["gap"]*base["gap"])).round(1)

base["reason_code"] = np.where(W["stale"]*base["stale"] >= W["gap"]*base["gap"],
                               "stale_visible_page", "low_ctr_for_position")
base["action"] = np.where(base["reason_code"] == "stale_visible_page",
                          "refresh_content", "rewrite_title_meta")
base["confidence"] = np.where((base["impressions_90d"] >= 500) & (base["sessions_90d"] >= 10), "high",
                       np.where(base["impressions_90d"] >= 250, "medium", "low"))

queue = base.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
cols = ["rank", "content_id", "score", "reason_code", "action", "confidence",
        "impressions_90d", "days_since_last_update", "avg_position", "ctr"]
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

json.dump({"n_ranked": int(len(queue)), "weights": W,
           "signal1_staleness_verdict": "FILL_IN", "signal2_ctr_gap_verdict": "FILL_IN"},
          open("work/outputs/baseline_summary.json", "w"), indent=2)
print(len(queue), "pages ranked and written to work/outputs/baseline_action_score.csv")

22006 pages ranked and written to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top = queue.head(20).copy()
wrong = {
  "stale_visible_page": "Wrong if the topic is evergreen (old date, still correct), or if the traffic drop is seasonal or absorbed by a sibling page.",
  "low_ctr_for_position": "Wrong if the low CTR comes from the search-results layout (answer boxes, AI summaries) and not from a weak title.",
}
top["what_would_make_it_wrong"] = top["reason_code"].map(wrong)
pd.set_option("display.max_colwidth", 120)
top[["rank", "score", "action", "reason_code", "confidence", "impressions_90d",
     "days_since_last_update", "avg_position", "ctr", "what_would_make_it_wrong"]]

,rank,score,action,reason_code,confidence,impressions_90d,days_since_last_update,avg_position,ctr,what_would_make_it_wrong
0,1,93.4,refresh_content,stale_visible_page,medium,208678,104,9.7,0.00,"Wrong if the topic is evergreen (old date, still correct), or if the traffic drop is seasonal or absorbed by a sibli..."
1,2,92.0,refresh_content,stale_visible_page,high,128068,104,2.2,0.01,"Wrong if the topic is evergreen (old date, still correct), or if the traffic drop is seasonal or absorbed by a sibli..."
2,3,91.8,refresh_content,stale_visible_page,high,79035,106,8.7,0.07,"Wrong if the topic is evergreen (old date, still correct), or if the traffic drop is seasonal or absorbed by a sibli..."
3,4,90.7,refresh_content,stale_visible_page,high,6526,106,17.4,0.00,"Wrong if the topic is evergreen (old date, still correct), or if the traffic drop is seasonal or absorbed by a sibli..."
4,5,90.1,refresh_content,stale_visible_page,high,134055,104,7.5,0.03,"Wrong if the topic is evergreen (old date, still correct), or if the traffic drop is seasonal or absorbed by a sibli..."
5,6,90.0,refresh_content,stale_visible_page,high,123469,104,8.0,0.03,"Wrong if the topic is evergreen (old date, still correct), or if the traffic drop is seasonal or absorbed by a sibli..."
6,7,89.8,refresh_content,stale_visible_page,high,83651,104,6.6,0.03,"Wrong if the topic is evergreen (old date, still correct), or if the traffic drop is seasonal or absorbed by a sibli..."
7,8,89.7,refresh_content,stale_visible_page,high,34100,104,6.2,0.02,"Wrong if the topic is evergreen (old date, still correct), or if the traffic drop is seasonal or absorbed by a sibli..."
8,9,89.6,refresh_content,stale_visible_page,high,22716,104,6.5,0.01,"Wrong if the topic is evergreen (old date, still correct), or if the traffic drop is seasonal or absorbed by a sibli..."
9,10,89.6,refresh_content,stale_visible_page,high,33286,104,4.6,0.02,"Wrong if the topic is evergreen (old date, still correct), or if the traffic drop is seasonal or absorbed by a sibli..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# weak picks: top-20 rows with thin evidence
print(top[top["confidence"] != "high"][["rank", "content_id", "score", "confidence", "impressions_90d"]])

# leakage check: what went into the score vs what must stay out
used_in_score = ["impressions_90d", "days_since_last_update", "avg_position", "ctr", "sessions_90d"]
banned = ["trend_direction", "declining", "health_score", "priority_score", "action_type", "refresh_tier"]
print("banned columns used in score:", [c for c in used_in_score if c in banned])  # must be []
print("product-flag columns present in queue:", [c for c in queue.columns if c in banned])

    rank            content_id  score confidence  impressions_90d
0      1  content_c8e9d6ab9013   93.4     medium           208678
10    11  content_825a9788af8d   89.5     medium            16786
12    13  content_8ba781dafa55   89.4     medium            16156
17    18  content_72fdb385e810   89.0     medium            19360
banned columns used in score: []
product-flag columns present in queue: ['trend_direction', 'declining']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.